### Import Dependencies

In [387]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [388]:
import os
os.chdir(r'C:\Kaggle_Competition\Playground\S6E5-F1-Pit-Stops')
from pathlib import Path
import numpy as np
import pandas as pd
import gc
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.inspection import permutation_importance
import mlflow
import shap

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier
from lightgbm import LGBMClassifier
from lightgbm import early_stopping
from xgboost import XGBClassifier
from xgboost.callback import EarlyStopping
from catboost import CatBoostClassifier
from pytabkit import RealMLP_TD_Classifier, TabM_D_Classifier
from sklearn.manifold import TSNE

# Preprocess
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import TargetEncoder, OneHotEncoder, StandardScaler, LabelEncoder, OrdinalEncoder
from feature_engine.outliers import Winsorizer

# Externals
from src.utils import *
from src.fe import *
import config
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

### Prepare Datasets

In [389]:
train = read_csv_file('data/raw/train.csv')
test = read_csv_file('data/raw/test.csv')
orig = read_csv_file('data/orig/f1_strategy_dataset_v4.csv')

train = reduce_mem_usage(train)
test = reduce_mem_usage(test)
orig = reduce_mem_usage(orig)

train_ids = train[config.ID_COL].values
test_ids = test[config.ID_COL].values

print(f'Shape of train data: {train.shape}')
print(f'Shape of test data: {test.shape}')
print(f'Shape of orig data: {orig.shape}')

Shape of train data: (439140, 16)
Shape of test data: (188165, 15)
Shape of orig data: (101371, 16)


### Logistic Regression

In [286]:
# Data
train_features = train.drop(['id', config.TARGET], axis=1)
target = train[config.TARGET].values
X_test = test.drop('id', axis=1)

In [287]:
def race_col(df):
    df['test_race'] = (
        df['race'] == 'Pre-Season Testing'
    ).astype('int8')

race_col(train_features)
race_col(X_test)

In [272]:
# Predictions
oof_predictions = np.zeros(len(train))
test_predictions = np.zeros(len(X_test))

# Experiment Setup
EXPERIMENT_NAME = "LR"
VERSION = "v2"
RUN_NAME = f"{EXPERIMENT_NAME}_{VERSION}_seed{config.SEED}"

mlflow.set_experiment(EXPERIMENT_NAME)

# CV
skf = StratifiedKFold(
    n_splits=config.N_FOLDS,
    shuffle=True,
    random_state=config.SEED
)

fold_loglosses = []
fold_aucs = []

with mlflow.start_run(run_name=RUN_NAME):

    for fold, (train_indices, valid_indices) in enumerate(
        skf.split(train_features, target),
        start=1
    ):

        X_train = train_features.iloc[train_indices]
        X_valid = train_features.iloc[valid_indices]

        y_train = target[train_indices]
        y_valid = target[valid_indices]

        preprocessor = ColumnTransformer(
            transformers=[
                ('scaler', StandardScaler(), config.BASE_NUM_COLS),
                ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore'), config.BASE_CAT_COLS)
            ],
            remainder='passthrough',
            n_jobs=-1
        )

        X_train_proc = preprocessor.fit_transform(X_train, y_train)
        X_valid_proc = preprocessor.transform(X_valid)
        X_test_proc = preprocessor.transform(X_test)

        model = LogisticRegression(**config.LR_PARAMS)

        model.fit(X_train_proc, y_train)

        fold_oof_predictions = model.predict_proba(X_valid_proc)[:, 1]
        fold_test_predictions = model.predict_proba(X_test_proc)[:, 1]

        oof_predictions[valid_indices] = fold_oof_predictions
        test_predictions += fold_test_predictions / config.N_FOLDS

        fold_logloss = log_loss(y_valid, fold_oof_predictions)
        fold_auc = roc_auc_score(y_valid, fold_oof_predictions)

        fold_loglosses.append(fold_logloss)
        fold_aucs.append(fold_auc)

        print("=" * 50)
        print(f"Fold {fold} LogLoss : {fold_logloss:.5f}")
        print(f"Fold {fold} ROC AUC : {fold_auc:.5f}")
        print("=" * 50)

    oof_logloss = log_loss(target, oof_predictions)
    oof_auc = roc_auc_score(target, oof_predictions)

    std_oof_logloss = np.std(fold_loglosses)
    std_oof_auc = np.std(fold_aucs)

    mlflow.log_metric("oof_logloss", oof_logloss)
    mlflow.log_metric("oof_auc", oof_auc)
    mlflow.log_metric("std_oof_logloss", std_oof_logloss)
    mlflow.log_metric("std_oof_auc", std_oof_auc)

    mlflow.log_params(config.LR_PARAMS) # Logging params

# Saving predictions
oof_proba_df = pd.DataFrame({'id': train_ids, 'PitNextLap': oof_predictions})
test_proba_df = pd.DataFrame({'id': test_ids, 'PitNextLap': test_predictions})

save_csv_file(oof_proba_df, Path(config.OOF_PROBA_DIR, f'{RUN_NAME}_oof_proba.csv'))
save_csv_file(test_proba_df, Path(config.TEST_PROBA_DIR, f'{RUN_NAME}_test_proba.csv'))

Fold 1 LogLoss : 0.48340
Fold 1 ROC AUC : 0.85617


KeyboardInterrupt: 

### HistGradientBoosting

In [ ]:
# Data
train_features = train.drop(['id', config.TARGET], axis=1)
target = train[config.TARGET].values
X_test = test.drop('id', axis=1)

In [ ]:
# Predictions
oof_predictions = np.zeros(len(train))
test_predictions = np.zeros(len(X_test))

# Experiment Setup
EXPERIMENT_NAME = "HISTGBM"
VERSION = "v1"
RUN_NAME = f"{EXPERIMENT_NAME}_{VERSION}"

mlflow.set_experiment(EXPERIMENT_NAME)

# CV
skf = StratifiedKFold(
    n_splits=config.N_FOLDS,
    shuffle=True,
    random_state=config.SEED
)

fold_loglosses = []
fold_aucs = []

with mlflow.start_run(run_name=RUN_NAME):

    for fold, (train_indices, valid_indices) in enumerate(
        skf.split(train_features, target),
        start=1
    ):
        X_train = train_features.iloc[train_indices]
        X_valid = train_features.iloc[valid_indices]

        y_train = target[train_indices]
        y_valid = target[valid_indices]

        preprocessor = ColumnTransformer(
            transformers=[
                ('encoder', TargetEncoder(target_type='binary', cv=5, shuffle=True, random_state=config.SEED), config.BASE_CAT_COLS)
            ],
            remainder='passthrough',
            n_jobs=-1
        )

        X_train_proc = preprocessor.fit_transform(X_train, y_train)
        X_valid_proc = preprocessor.transform(X_valid)
        X_test_proc = preprocessor.transform(X_test)

        model = HistGradientBoostingClassifier(**config.HISTGBM_PARAMS)

        model.fit(X_train_proc, y_train)

        fold_oof_predictions = model.predict_proba(X_valid_proc)[:, 1]
        fold_test_predictions = model.predict_proba(X_test_proc)[:, 1]

        oof_predictions[valid_indices] = fold_oof_predictions
        test_predictions += fold_test_predictions / config.N_FOLDS

        fold_logloss = log_loss(y_valid, fold_oof_predictions)
        fold_auc = roc_auc_score(y_valid, fold_oof_predictions)

        fold_loglosses.append(fold_logloss)
        fold_aucs.append(fold_auc)

        print("=" * 50)
        print(f"Fold {fold} LogLoss : {fold_logloss:.5f}")
        print(f"Fold {fold} ROC AUC : {fold_auc:.5f}")
        print("=" * 50)

    oof_logloss = log_loss(target, oof_predictions)
    oof_auc = roc_auc_score(target, oof_predictions)

    std_oof_logloss = np.std(fold_loglosses)
    std_oof_auc = np.std(fold_aucs)

    mlflow.log_metric("oof_logloss", oof_logloss)
    mlflow.log_metric("oof_auc", oof_auc)
    mlflow.log_metric("std_oof_logloss", std_oof_logloss)
    mlflow.log_metric("std_oof_auc", std_oof_auc)

    mlflow.log_params(config.HISTGBM_PARAMS) # Logging params

# Saving predictions
oof_proba_df = pd.DataFrame({'id': train_ids, 'PitNextLap': oof_predictions})
test_proba_df = pd.DataFrame({'id': test_ids, 'PitNextLap': test_predictions})

save_csv_file(oof_proba_df, Path(config.OOF_PROBA_DIR, f'{RUN_NAME}_oof_proba.csv'))
save_csv_file(test_proba_df, Path(config.TEST_PROBA_DIR, f'{RUN_NAME}_test_proba.csv'))

Fold 1 LogLoss : 0.30112
Fold 1 ROC AUC : 0.94608
Fold 2 LogLoss : 0.30216
Fold 2 ROC AUC : 0.94464
Fold 3 LogLoss : 0.30091
Fold 3 ROC AUC : 0.94549
Fold 4 LogLoss : 0.30276
Fold 4 ROC AUC : 0.94436
Fold 5 LogLoss : 0.29760
Fold 5 ROC AUC : 0.94559


### LGBM

In [390]:
# Data
train_features = train.drop(['id', config.TARGET], axis=1)
target = train[config.TARGET].values
X_test = test.drop('id', axis=1)
orig_features = orig.drop([config.TARGET, 'normalized_tyrelife'], axis=1)

In [391]:
# Winsorization
winsor = Winsorizer(
    capping_method='iqr',
    tail='both',
    fold=5,
    variables = [
    "tyrelife",
    "laptime (s)",
    "laptime_delta",
    "cumulative_degradation",
    "raceprogress"
    ]
)

train_features = winsor.fit_transform(train_features)
X_test = winsor.transform(X_test)
orig_features = winsor.transform(orig_features)

In [392]:
# Frequency Encoding
for col in config.BASE_CAT_COLS:

    freq_map = train_features[col].value_counts(normalize=True).to_dict()

    train_features[f"{col}_freq"] = train_features[col].map(freq_map)
    X_test[f"{col}_freq"] = X_test[col].map(freq_map).fillna(0)
    orig_features[f"{col}_freq"] = orig_features[col].map(freq_map).fillna(0)

# Ordinal Encoding
encoder = OrdinalEncoder(
    handle_unknown='use_encoded_value',
    unknown_value=-1
)

train_features[config.BASE_CAT_COLS] = encoder.fit_transform(
    train_features[config.BASE_CAT_COLS].astype(str)
).astype("int32")

X_test[config.BASE_CAT_COLS] = encoder.transform(
    X_test[config.BASE_CAT_COLS].astype(str)
).astype("int32")

orig_features[config.BASE_CAT_COLS] = encoder.transform(
    orig_features[config.BASE_CAT_COLS].astype(str)
).astype("int32")

In [393]:
# Stratification labels
stratify_labels = (
    target.astype(str) + "_" + train_features["year"].astype(str)
)

In [ ]:
# Predictions
oof_predictions = np.zeros(len(train))
test_predictions = np.zeros(len(X_test))

# Experiment Setup
EXPERIMENT_NAME = "LGBM"
VERSION = "v6"
RUN_NAME = f"{EXPERIMENT_NAME}_{VERSION}_seed{config.SEED}"

mlflow.set_experiment(EXPERIMENT_NAME)

# CV
skf = StratifiedKFold(
    n_splits=config.N_FOLDS,
    shuffle=True,
    random_state=config.SEED
)

fold_loglosses = []
fold_aucs = []

with mlflow.start_run(run_name=RUN_NAME):

    for fold, (train_indices, valid_indices) in enumerate(
        skf.split(train_features, stratify_labels),
        start=1
    ):
        X_train = train_features.iloc[train_indices]
        X_valid = train_features.iloc[valid_indices]

        y_train = target[train_indices]
        y_valid = target[valid_indices]

        preprocessor = ColumnTransformer(
            transformers=[
                #('encoder', TargetEncoder(target_type='binary', cv=5, shuffle=True, random_state=config.SEED), config.BASE_CAT_COLS)
            ],
            remainder='passthrough',
            n_jobs=-1
        )

        X_train_proc = preprocessor.fit_transform(X_train, y_train)
        X_valid_proc = preprocessor.transform(X_valid)
        X_test_proc = preprocessor.transform(X_test)

        model = LGBMClassifier(**config.LGBM_PARAMS)

        model.fit(X_train_proc, y_train, eval_set=[(X_valid_proc, y_valid)], callbacks=[early_stopping(100)])

        fold_oof_predictions = model.predict_proba(X_valid_proc)[:, 1]
        fold_test_predictions = model.predict_proba(X_test_proc)[:, 1]

        oof_predictions[valid_indices] = fold_oof_predictions
        test_predictions += fold_test_predictions / config.N_FOLDS

        fold_logloss = log_loss(y_valid, fold_oof_predictions)
        fold_auc = roc_auc_score(y_valid, fold_oof_predictions)

        fold_loglosses.append(fold_logloss)
        fold_aucs.append(fold_auc)

        print("=" * 50)
        print(f"Fold {fold} LogLoss : {fold_logloss:.5f}")
        print(f"Fold {fold} ROC AUC : {fold_auc:.5f}")
        print("=" * 50)

    oof_logloss = log_loss(target, oof_predictions)
    oof_auc = roc_auc_score(target, oof_predictions)

    std_oof_logloss = np.std(fold_loglosses)
    std_oof_auc = np.std(fold_aucs)

    mlflow.log_metric("oof_logloss", oof_logloss)
    mlflow.log_metric("oof_auc", oof_auc)
    mlflow.log_metric("std_oof_logloss", std_oof_logloss)
    mlflow.log_metric("std_oof_auc", std_oof_auc)

    mlflow.log_params(config.LGBM_PARAMS) # Logging params

# Saving predictions
oof_proba_df = pd.DataFrame({'id': train_ids, 'PitNextLap': oof_predictions})
test_proba_df = pd.DataFrame({'id': test_ids, 'PitNextLap': test_predictions})

save_csv_file(oof_proba_df, Path(config.OOF_PROBA_DIR, f'{RUN_NAME}_oof_proba.csv'))
save_csv_file(test_proba_df, Path(config.TEST_PROBA_DIR, f'{RUN_NAME}_test_proba.csv'))

Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[8930]	valid_0's auc: 0.949537


In [ ]:
"""
==================================================
Fold 1 LogLoss : 0.28236
Fold 1 ROC AUC : 0.94970
==================================================
Fold 2 LogLoss : 0.28547
Fold 2 ROC AUC : 0.94810
==================================================
Fold 3 LogLoss : 0.28435
Fold 3 ROC AUC : 0.94873
==================================================
Fold 4 LogLoss : 0.28671
Fold 4 ROC AUC : 0.94782
==================================================
Fold 5 LogLoss : 0.28058
Fold 5 ROC AUC : 0.94934
=================================================="""

### XGBoost

In [214]:
# Data
train_features = train.drop(['id', config.TARGET], axis=1)
target = train[config.TARGET].values
X_test = test.drop('id', axis=1)

orig_features = orig.drop([config.TARGET], axis=1)
orig_features = orig_features.dropna(axis=0)

In [215]:
# Create n-gram categorical features
train_features, bi_cols = create_cat_bigrams(
    train_features,
    config.BASE_CAT_COLS
)

train_features, tri_cols = create_cat_trigrams(
    train_features,
    config.BASE_CAT_COLS
)

X_test, _ = create_cat_bigrams(X_test, config.BASE_CAT_COLS)
X_test, _ = create_cat_trigrams(X_test, config.BASE_CAT_COLS)

orig_features, _ = create_cat_bigrams(orig_features, config.BASE_CAT_COLS)
orig_features, _ = create_cat_trigrams(orig_features, config.BASE_CAT_COLS)

# All generated categorical columns
all_ngram_cols = bi_cols + tri_cols

for col in all_ngram_cols:

    combined = pd.concat(
        [
            train_features[col],
            X_test[col],
            orig_features[col]
        ]
    ).astype(str)

    codes, _ = pd.factorize(combined)

    n_train = len(train_features)
    n_test = len(X_test)

    train_features[col] = codes[:n_train].astype("int32")
    X_test[col] = codes[n_train:n_train+n_test].astype("int32")
    orig_features[col] = codes[n_train+n_test:].astype("int32")

Added 3 bigram features
Added 1 trigram features
Added 3 bigram features
Added 1 trigram features
Added 3 bigram features
Added 1 trigram features


In [216]:
for col in config.BASE_CAT_COLS:

    combined = pd.concat(
        [
            train_features[col].astype(str),
            X_test[col].astype(str),
            orig_features[col].astype(str)
        ]
    )

    codes, _ = pd.factorize(combined)

    n_train = len(train_features)
    n_test = len(X_test)

    train_features[col] = pd.Categorical(codes[:n_train])
    X_test[col] = pd.Categorical(codes[n_train:n_train+n_test])
    orig_features[col] = pd.Categorical(codes[n_train+n_test:])

In [217]:
TE_COLS = all_ngram_cols + config.BASE_CAT_COLS

In [ ]:
# Predictions
oof_predictions = np.zeros(len(train))
test_predictions = np.zeros(len(X_test))

# Experiment Setup
EXPERIMENT_NAME = "XGB"
VERSION = "v7"
RUN_NAME = f"{EXPERIMENT_NAME}_{VERSION}_{config.SEED}"

mlflow.set_experiment(EXPERIMENT_NAME)

# CV
skf = StratifiedKFold(
    n_splits=config.N_FOLDS,
    shuffle=True,
    random_state=config.SEED
)

fold_loglosses = []
fold_aucs = []

with mlflow.start_run(run_name=RUN_NAME):

    for fold, (train_indices, valid_indices) in enumerate(
        skf.split(train_features, target),
        start=1
    ):
        X_train = train_features.iloc[train_indices]
        X_valid = train_features.iloc[valid_indices]

        y_train = target[train_indices]
        y_valid = target[valid_indices]

        preprocessor = ColumnTransformer(
            transformers=[
                ('encoder', TargetEncoder(cv=5, target_type='binary', shuffle=True, random_state=config.SEED), TE_COLS)
            ],
            remainder='passthrough',
            n_jobs=-1
        )

        X_train_proc = preprocessor.fit_transform(X_train, y_train)
        X_valid_proc = preprocessor.transform(X_valid)
        X_test_proc = preprocessor.transform(X_test)

        early_stop = EarlyStopping(rounds=100, metric_name='auc', save_best=True)
        model = XGBClassifier(**config.XGBM_PARAMS)
        model.fit(X_train_proc, y_train, eval_set=[(X_valid_proc, y_valid)], verbose=1000)

        fold_oof_predictions = model.predict_proba(X_valid_proc)[:, 1]
        fold_test_predictions = model.predict_proba(X_test_proc)[:, 1]

        oof_predictions[valid_indices] = fold_oof_predictions
        test_predictions += fold_test_predictions / config.N_FOLDS

        fold_logloss = log_loss(y_valid, fold_oof_predictions)
        fold_auc = roc_auc_score(y_valid, fold_oof_predictions)

        fold_loglosses.append(fold_logloss)
        fold_aucs.append(fold_auc)

        print("=" * 50)
        print(f"Fold {fold} LogLoss : {fold_logloss:.5f}")
        print(f"Fold {fold} ROC AUC : {fold_auc:.5f}")
        print("=" * 50)

    oof_logloss = log_loss(target, oof_predictions)
    oof_auc = roc_auc_score(target, oof_predictions)

    std_oof_logloss = np.std(fold_loglosses)
    std_oof_auc = np.std(fold_aucs)

    mlflow.log_metric("oof_logloss", oof_logloss)
    mlflow.log_metric("oof_auc", oof_auc)
    mlflow.log_metric("std_oof_logloss", std_oof_logloss)
    mlflow.log_metric("std_oof_auc", std_oof_auc)

    mlflow.log_params(config.XGBM_PARAMS) # Logging params

# Saving predictions
oof_proba_df = pd.DataFrame({'id': train_ids, 'PitNextLap': oof_predictions})
test_proba_df = pd.DataFrame({'id': test_ids, 'PitNextLap': test_predictions})

save_csv_file(oof_proba_df, Path(config.OOF_PROBA_DIR, f'{RUN_NAME}_oof_proba.csv'))
save_csv_file(test_proba_df, Path(config.TEST_PROBA_DIR, f'{RUN_NAME}_test_proba.csv'))